Code Structure Overview (Current and Planned)

1. Parameters dictionary (params):
   Stores all physical and geometric parameters required by the model:
     - mass m
     - gravity g
     - thrust coefficient kf
     - drag torque coefficient km
     - arm length L
     - moments of inertia (Ixx, Iyy, Izz)
   Additional parameters (motor limits, noise, battery voltage curves, etc.)
   may be added later for the extra credit “reality check” using actual hardware
   specifications.

2. quad_eom (Equations of Motion):
   Implements the full 6-DOF rigid-body dynamics of the quadcopter.
   Inputs: current time t, current state vector u, parameter dictionary params,
           and a chosen controller function ctrl.
   Workflow:
     - Calls ctrl(t, u, params) to compute rotor speeds (w1–w4).
     - Computes total thrust FT and body torques (tau_x, tau_y, tau_z)
       produced by the four rotors.
     - Evaluates translational and rotational equations of motion derived in
       the report.
     - Returns du/dt, the time derivative of the 12-state vector.
   Note: quad_eom does not update the state itself. The numerical integrator
   handles state propagation using the returned derivative.

3. Control functions (ctrl_hover, ctrl_circle, ctrl_traj):
   One controller is implemented per performance goal:
       - Hover regulation
       - Circular trajectory tracking
       - Full multi-stage trajectory with yaw rotation and landing
   Each controller:
       - Computes state errors relative to desired references (altitude,
         position, attitude, yaw, etc.)
       - Produces desired total thrust FT and control torques tau_x, tau_y,
         tau_z
       - Calls the mixer to convert these physical commands into individual
         rotor speeds (w1–w4)
       - Returns the rotor speeds to quad_eom.

4. Mixer:
   A common actuator-allocation module used by all control functions.
   Converts desired physical forces/moments (FT, tau_x, tau_y, tau_z) into
   rotor speeds by inverting the linear relationship between rotor thrusts and
   the net force/torque vector.
   This ensures:
       - Controllers remain clean and intuitive (they work in physical units)
       - All actuator logic stays centralized
       - The same mixer works unchanged for all test cases

5. Time integration / Approximation function:
   Numerical integrator responsible for evolving the state in time.
   Options:
       - Custom implementation (Euler, RK4) as learned in TAM 370
       - Using scipy.integrate.solve_ivp for reliability and step control
   Inputs: quad_eom, initial state u0, time span, and step size.
   Output: full time history of u(t) for analysis and plotting.
   Not yet implemented.

6. Driver function:
   Highest-level function that sets up and runs a simulation.
   Responsibilities:
       - Define params
       - Define initial state u0
       - Choose which controller to use (hover/circle/trajectory)
       - Choose integrator settings (duration, dt, solver)
       - Call the time integrator
       - Gather and return simulation results
   Not yet implemented.

7. Plotting functions:
   Produce summary plots from simulation results, including:
       - Height vs. time
       - x-y trajectory (for circle and multi-step test cases)
       - Euler angle evolution
       - Rotor speeds over time
   These will be used to evaluate whether each performance goal is satisfied.

Current status:
   The hover controller function (ctrl_hover) is implemented but untested.
   Mixer and quad_eom are implemented.
   Integration, driver routines, and plotting functions are not implemented


In [ ]:
import numpy as np

params = {
    "m":   1.0,        # mass [kg]
    "g":   9.81,       # gravity [m/s^2]

    # thrust + moment coefficients
    "kf":  1e-5,       # thrust coefficient [N/(rad/s)^2]
    "km":  2e-6,       # drag/moment coefficient [Nm/(rad/s)^2]

    # geometry
    "L":   0.2,        # arm length [m]

    # inertia (example values; replace with your actual inertia matrix)
    "Ixx": 0.005,      # moment of inertia about x [kg·m^2]
    "Iyy": 0.005,      # moment of inertia about y [kg·m^2]
    "Izz": 0.009       # moment of inertia about z [kg·m^2]
}


def quad_eom(t, u, params, ctrl):
    """
    u = [x, y, z, vx, vy, vz, phi, theta, psi, p, q, r]
    ctrl(t, u, params) -> rotor speeds omega1..omega4
    """
    m   = params["m"]
    g   = params["g"]
    kf  = params["kf"]
    km  = params["km"]
    L   = params["L"]
    Ixx = params["Ixx"]
    Iyy = params["Iyy"]
    Izz = params["Izz"]

    x, y, z, vx, vy, vz, phi, theta, psi, p, q, r = u

    # --- control: motor speeds ---
    w1, w2, w3, w4 = ctrl(t, u, params)

    # total thrust and torques
    FT = kf * (w1**2 + w2**2 + w3**2 + w4**2)
    tau_x = kf * L * (w3**2 + w4**2 - w1**2 - w2**2)
    tau_y = kf * L * (w2**2 + w3**2 - w1**2 - w4**2)
    tau_z = km * (w1**2 + w3**2 - w2**2 - w4**2)

    # rotation-related shorthands
    cphi, sphi   = np.cos(phi), np.sin(phi)
    cth, sth     = np.cos(theta), np.sin(theta)
    cpsi, spsi   = np.cos(psi), np.sin(psi)

    # translational accelerations from your EOMs
    ax = FT/m * ( spsi*sphi - cpsi*sth*cphi )
    ay = FT/m * ( cpsi*(-sphi) - spsi*sth*cphi )
    az = FT/m * ( cth*cphi ) - g

    # rotational dynamics (Euler equations)
    p_dot = (tau_x + (Iyy - Izz)*q*r) / Ixx
    q_dot = (tau_y + (Izz - Ixx)*p*r) / Iyy
    r_dot = (tau_z + (Ixx - Iyy)*p*q) / Izz

    # Euler angle kinematics (standard T(phi,theta) matrix)
    phi_dot   = p + np.tan(theta)*(q*np.sin(phi) + r*np.cos(phi))
    theta_dot = q*np.cos(phi) - r*np.sin(phi)
    psi_dot   = (q*np.sin(phi) + r*np.cos(phi)) / np.cos(theta)

    return np.array([
        vx, vy, vz,        # x_dot, y_dot, z_dot
        ax, ay, az,        # vx_dot, vy_dot, vz_dot
        phi_dot, theta_dot, psi_dot,
        p_dot, q_dot, r_dot
    ])

import numpy as np

def mixer(FT, tau_x, tau_y, tau_z, params):
    """
    Convert desired total thrust and body torques into individual rotor speeds.

    Solves:
        [ kf   kf   kf   kf  ] [w1^2]   [ FT    ]
        [-kfL  kfL  kfL -kfL ] [w2^2] = [ tau_x ]
        [-kfL -kfL  kfL  kfL ] [w3^2]   [ tau_y ]
        [ km  -km   km  -km  ] [w4^2]   [ tau_z ]

    Returns rotor speeds w1..w4 (rad/s).
    """
    kf  = params["kf"]
    km  = params["km"]
    L   = params["L"]

    M = np.array([
        [ kf,      kf,      kf,      kf     ],
        [-kf*L,    kf*L,    kf*L,   -kf*L   ],
        [-kf*L,   -kf*L,    kf*L,    kf*L   ],
        [ km,     -km,      km,     -km     ]
    ])

    rhs = np.array([FT, tau_x, tau_y, tau_z])

    # Solve for squared speeds and guard against tiny negative values
    w_sq = np.linalg.solve(M, rhs)
    w_sq_clipped = np.clip(w_sq, 0.0, None)

    w = np.sqrt(w_sq_clipped)
    return w[0], w[1], w[2], w[3]


def ctrl_hover(t, u, params):
    """
    Hover controller: hold position at (0, 0, z_ref) with level attitude and fixed yaw.

    Inputs
    ------
    t : float
        Time (unused here but kept for compatibility).
    u : array_like, shape (12,)
        State vector [x, y, z, vx, vy, vz, phi, theta, psi, p, q, r].
    params : dict
        Model and controller parameters (must contain m, g, kf, km, L).

    Returns
    -------
    w1, w2, w3, w4 : float
        Rotor speeds [rad/s] for the four motors.
    """

    m = params["m"]
    g = params["g"]

    # Unpack state
    x, y, z, vx, vy, vz, phi, theta, psi, p, q, r = u

    # -------- Altitude control (PD on z) --------
    z_ref = 1.0  # desired hover height [m]
    ez   = z_ref - z
    evz  = -vz

    Kp_z = 10.0
    Kd_z = 5.0

    # Desired total thrust (nominal mg plus correction)
    FT = m * g + Kp_z * ez + Kd_z * evz

    # Optionally clamp FT to reasonable range
    FT_min = 0.2 * m * g
    FT_max = 2.0 * m * g
    FT = np.clip(FT, FT_min, FT_max)

    # -------- Attitude control (keep level, hold yaw) --------
    phi_ref   = 0.0   # level roll
    theta_ref = 0.0   # level pitch
    psi_ref   = 0.0   # desired yaw (can change if you want)

    e_phi   = phi_ref   - phi
    e_theta = theta_ref - theta
    e_psi   = psi_ref   - psi

    Kp_phi, Kd_phi = 4.0, 1.5
    Kp_th,  Kd_th  = 4.0, 1.5
    Kp_psi, Kd_psi = 3.0, 1.0

    tau_x = Kp_phi * e_phi   + Kd_phi * (-p)
    tau_y = Kp_th  * e_theta + Kd_th  * (-q)
    tau_z = Kp_psi * e_psi   + Kd_psi * (-r)

    # -------- Mix to rotor speeds --------
    w1, w2, w3, w4 = mixer(FT, tau_x, tau_y, tau_z, params)
    return w1, w2, w3, w4

